In [75]:
from IPython.display import Audio
from pathlib import Path 
import librosa 
import json
import pickle
import numpy as np 
import soundfile as sf
import scipy.signal as sig


import sys
# get audio utils
sys.path.append('/om2/user/imgriff/datasets/spatial_audio_pipeline/utils')
import util_audio



In [19]:

stim_path = Path('/mindhive/mcdermott/www/mturk_stimuli/imgriff/single_word_rec/pilot/giga_200_words_original')
eg = stim_path / 'Can.wav'

# Audio(eg)

In [20]:
wav, sr = librosa.load(eg, sr=None)

In [21]:
wav.shape[0] / sr, sr

(1.75, 44100)

In [22]:
giga_words = open('corpus/gigaspeech_top_200.txt', 'r').read().splitlines()
giga_words = [word.lower() for word in giga_words]

In [26]:
## Get dicts of common voice words &  word-class ints 

cv_label_map = pickle.load(open('../../datasets/commonvoice_9/en/cv_word_int_label_dict.pkl', 'rb'))

cv_prolific_dict = json.load(open('../../datasets/commonvoice_9/en/cv_model_and_participant_eval_word_dict.json', 'rb'))
cv_expmt_words = set(cv_prolific_dict['dictionary'])


In [31]:
# Get valid gigaspeech word recordings to use as catch trials
giga_words_to_use = cv_expmt_words.intersection(giga_words)
print(len(giga_words_to_use))

giga_to_cv_wordclass_map = {word:cv_label_map[word] for word in giga_words_to_use}


48


In [69]:
def ramp_hann(x, ramp_dur_ms, samplerate=20_000):
    stim_dur_smp = x.shape[0] # N taps of x
    ramp_dur_smp =  np.floor(ramp_dur_ms * samplerate / 1000).astype('int')
    assert stim_dur_smp > (2*ramp_dur_smp), 'Ramps cannot be longer than the stimulus duration'
    
    # calc window
    # https://stackoverflow.com/questions/56485663/hanning-window-values-doesnt-match-in-python-and-matlab
    win = sig.hann((2 * ramp_dur_smp) + 2)[1:-1]
    
    # Beginning of windowed stimulus
    start_win = win[ : ramp_dur_smp ] * x[ : ramp_dur_smp]
    
    # Middle part (steady state)
    steady_win = x[ramp_dur_smp : stim_dur_smp-ramp_dur_smp]
    
    # Final part of windowed stimulus
    end_win = win[ramp_dur_smp : ramp_dur_smp*2] * x[stim_dur_smp-ramp_dur_smp:stim_dur_smp]

    return np.hstack([start_win, steady_win, end_win])

In [76]:
# Resample and save to cv experiment directory

giga_word_files = stim_path.glob('*.wav')

path_to_cv = Path('/mindhive/mcdermott/www/imgriff/msjspsych/commonvoice_word_recognition/stim/catch_trial_stim')

OUT_SR = 20_000
dur_in_frames = int(2 * OUT_SR) # 2 sec in frames
# resample and save each word using class ix as file name. eg: {word_ix}_{versionx}.wav
for word_file in giga_word_files:
    word = word_file.stem.lower()
    if word in giga_to_cv_wordclass_map:
        # get word int 
        class_ix = giga_to_cv_wordclass_map[word]
        # load audio
        wav, sr = librosa.load(word_file, sr=OUT_SR, res_type='soxr_vhq')
        #ensure is 2 seconds 
        dur_diff = dur_in_frames - len(wav)
        if dur_diff > 0:
            wav = np.pad(wav, (int(np.floor(dur_diff / 2)), int(np.ceil(dur_diff / 2))), 'constant')
        else:
            wav = wav[int(np.floor(dur_diff / 2)) : int(np.ceil(dur_diff / 2))]
        # window edges to remove pops 
        wav = ramp_hann(wav, 10, samplerate=OUT_SR)
        # set to 60dB SPL 
        wav = util_audio.set_dBSPL(wav, dBSPL=60, mean_subtract=True)
        # set out name and save
        out_name = path_to_cv / f"{class_ix:03}_000.wav"
        if not out_name.parent.exists():
            out_name.parent.mkdir(exist_ok=True, parents=True)
        sf.write(out_name.as_posix(), wav, OUT_SR, subtype='PCM_24') 

        # break



In [57]:
wav.shape[0] / OUT_SR

2.0

In [73]:
list(giga_to_cv_wordclass_map.values())

[250,
 55,
 152,
 214,
 555,
 43,
 142,
 39,
 495,
 204,
 149,
 423,
 80,
 140,
 522,
 122,
 60,
 155,
 40,
 295,
 117,
 309,
 71,
 73,
 608,
 321,
 254,
 44,
 93,
 714,
 138,
 790,
 186,
 150,
 145,
 56,
 288,
 285,
 366,
 562,
 391,
 193,
 95,
 86,
 789,
 464,
 106,
 301]